# Detection Fine-Tune Dataset Builder

Single notebook pipeline:
1. Extract candidate frames from videos
2. Select diverse + low-confidence frames using embeddings
3. Seed labels with YOLO26 predictions
4. Human review/redraw and export
5. Copy alongside coco datasets

In [ ]:
import sys
import os
import random
import hashlib
from functools import cache
from dataclasses import dataclass
from pathlib import Path
from shutil import copy2

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import utils
from config import settings

device = utils.get_best_device()

In [ ]:
# Define constants
TARGET_LABEL_FRAMES = 10
YOLO_CONF_FLOOR = 0.05
ALPHA_UNCERTAINTY = 0.5
ALPHA_BALANCE = 0.2
TRAIN_SPLIT = 0.8
CLASS_NAMES = {0: "person", 1: "cat"}
SOURCE_SPLITS = ["train", "val"]

In [ ]:
# Define paths
MOCK_INPUTS = PROJECT_ROOT / "datasets" / "mock_inputs"
RAW_VIDEO = PROJECT_ROOT / "datasets" / "raw_video"
EXPORT_ROOT = PROJECT_ROOT / "datasets" / "finetune_data"
CROPPED_EXPORT_ROOT = PROJECT_ROOT / "datasets" / "finetune_data_cropped"

In [ ]:
# Class/function definitions
@dataclass
class Candidate:
    """Store frame-level data, labels, uncertainty, and embedding for selection."""

    frame_id: int
    video_name: str
    frame_idx: int
    image_bgr: np.ndarray
    labels_cxcywhn: list
    uncertainty: float
    embedding: np.ndarray


def l2_normalize(vec: np.ndarray) -> np.ndarray:
    """Return a unit-length copy of a vector with numerical stability."""
    n = np.linalg.norm(vec)
    return vec / (n + 1e-12)


@torch.inference_mode()
def compute_embedding(
    image_bgr: np.ndarray, preprocess: transforms.Compose, embedder: torch.nn.Module
) -> np.ndarray:
    """Generate a normalized image embedding from the frame using ResNet features."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(image_rgb)
    x = preprocess(pil).unsqueeze(0).to(device)
    feat = embedder(x).squeeze().detach().cpu().numpy().astype(np.float32)
    return l2_normalize(feat)


def get_image_hash(image_bgr: np.ndarray) -> str:
    """Return a stable hash key for frame/image bytes."""
    return hashlib.sha1(image_bgr.tobytes()).hexdigest()


@cache
def _embedding_from_bytes(
    image_bytes: bytes, shape: tuple[int, int, int]
) -> np.ndarray:
    """Compute and cache embedding for a byte-identical image."""
    image_bgr = np.frombuffer(image_bytes, dtype=np.uint8).reshape(shape).copy()
    return compute_embedding(
        image_bgr, preprocess=embedding_preprocess, embedder=embedding_model
    )


def get_embedding_cached(image_bgr: np.ndarray) -> np.ndarray:
    """Get embedding from unbounded functools cache."""
    return _embedding_from_bytes(image_bgr.tobytes(), tuple(image_bgr.shape))


@cache
def _yolo_prediction_from_bytes(
    image_bytes: bytes,
    shape: tuple[int, int, int],
    imgsz: int,
    conf: float,
    max_det: int,
):
    """Compute and cache YOLO predictions and parsed labels for a byte-identical image."""
    image_bgr = np.frombuffer(image_bytes, dtype=np.uint8).reshape(shape).copy()
    frame_wh = tuple(reversed(image_bgr.shape[:2]))
    pred = suggestor_model.predict(
        image_bgr, imgsz=imgsz, conf=conf, max_det=max_det, verbose=False
    )[0]
    boxes = pred.boxes

    if boxes is None or len(boxes) == 0:
        bboxes, classes, confs = [], [], []
    else:
        classes = boxes.cls.detach().cpu().numpy().astype(int)
        confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
        bboxes = [
            utils.Bbox(xyxy=xyxy, frame_wh=frame_wh)
            for xyxy in boxes.xyxy.detach().cpu().numpy()
        ]

    labels = []
    max_conf = 0.0
    for bbox, conf_score, cls_id in zip(bboxes, confs, classes):
        if cls_id not in CLASS_NAMES:
            continue
        labels.append((cls_id, *bbox.cxcywhn))
        max_conf = max(max_conf, conf_score)

    return {
        "bboxes": bboxes,
        "classes": classes,
        "confs": confs,
        "labels": labels,
        "max_conf": max_conf,
    }


def get_yolo_prediction_cached(
    image_bgr: np.ndarray, imgsz: int, conf: float, max_det: int
):
    """Get YOLO prediction payload from unbounded functools cache."""
    return _yolo_prediction_from_bytes(
        image_bgr.tobytes(),
        tuple(image_bgr.shape),
        imgsz,
        conf,
        max_det,
    )


def draw_cxcywhn(image_bgr, labels_cxcywhn):
    """Render normalized YOLO boxes and class labels onto an image."""
    img = image_bgr.copy()
    frame_wh = tuple(reversed(image_bgr.shape[:2]))
    for cls_id, xc, yc, bw, bh in labels_cxcywhn:
        x1, y1, x2, y2 = utils.Bbox(cxcywhn=(xc, yc, bw, bh), frame_wh=frame_wh).xyxy
        colour = utils.OBJECT_COLOUR_MAP[CLASS_NAMES[cls_id]]
        cv2.rectangle(img, (x1, y1), (x2, y2), colour, 2)
        cv2.putText(
            img,
            CLASS_NAMES.get(cls_id, str(cls_id)),
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            colour,
            1,
            cv2.LINE_AA,
        )
    return img


def parse_yolo_txt(path):
    """Read YOLO label rows from a text file into structured tuples."""
    labels = []
    if not path.exists():
        return labels
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 5:
                labels.append(
                    (int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4]))
                )
    return labels


def draw_boxes_interactive(image, window_name="draw boxes"):
    """Open an interactive UI for drawing and editing bounding boxes."""
    boxes = []
    class_state = {"cls": 1}
    drawing = {"active": False, "x0": 0, "y0": 0, "x1": 0, "y1": 0}
    base = image.copy()

    def on_mouse(event, x, y, flags, param):
        """Handle mouse interactions for creating rectangle annotations."""
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing["active"] = True
            drawing["x0"], drawing["y0"] = x, y
            drawing["x1"], drawing["y1"] = x, y
        elif event == cv2.EVENT_MOUSEMOVE and drawing["active"]:
            drawing["x1"], drawing["y1"] = x, y
        elif event == cv2.EVENT_LBUTTONUP and drawing["active"]:
            drawing["active"] = False
            drawing["x1"], drawing["y1"] = x, y
            x0, y0 = drawing["x0"], drawing["y0"]
            x1, y1 = drawing["x1"], drawing["y1"]
            x0, x1 = sorted([x0, x1])
            y0, y1 = sorted([y0, y1])
            if (x1 - x0) > 2 and (y1 - y0) > 2:
                boxes.append((x0, y0, x1, y1, class_state["cls"]))

    cv2.namedWindow(window_name)
    cv2.setMouseCallback(window_name, on_mouse)

    while True:
        canvas = base.copy()
        for x0, y0, x1, y1, cls_id in boxes:
            colour = utils.OBJECT_COLOUR_MAP[CLASS_NAMES[cls_id]]
            cv2.rectangle(canvas, (x0, y0), (x1, y1), colour, 2)
            cv2.putText(
                canvas,
                CLASS_NAMES.get(cls_id, str(cls_id)),
                (x0, max(15, y0 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                colour,
                1,
                cv2.LINE_AA,
            )
        if drawing["active"]:
            colour = utils.OBJECT_COLOUR_MAP[CLASS_NAMES[class_state["cls"]]]
            cv2.rectangle(
                canvas,
                (drawing["x0"], drawing["y0"]),
                (drawing["x1"], drawing["y1"]),
                colour,
                2,
            )

        current = CLASS_NAMES.get(class_state["cls"], str(class_state["cls"]))
        tip = f"Class:{current} | 0=person 1=cat | Drag draw | u undo | x clear | Enter/Space save | Esc cancel"
        cv2.putText(
            canvas,
            tip,
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 220, 0),
            2,
            cv2.LINE_AA,
        )
        cv2.imshow(window_name, canvas)

        k = cv2.waitKey(20) & 0xFF
        if k == ord("0"):
            class_state["cls"] = 0
        if k == ord("1"):
            class_state["cls"] = 1
        if k in (13, 32):  # Enter or Space
            cv2.destroyWindow(window_name)
            return boxes
        if k == 27:  # Esc
            cv2.destroyWindow(window_name)
            return None
        if k == ord("u") and boxes:
            boxes.pop()
        if k == ord("x"):
            boxes = []


def count_lines(path):
    """Count non-empty lines in a text file."""
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f if _.strip())


def random_aspect_crop_bbox(w, h):
    """Sample a random crop and expand it to source aspect ratio."""
    cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
    box_h = random.randint(max(2, int(h * 0.1)), max(2, int(h * 0.2)))
    half_h = max(1, box_h // 2)

    x_min = max(0, cx - half_h)
    x_max = min(w - 1, cx + half_h)
    y_min = max(0, cy - half_h)
    y_max = min(h - 1, cy + half_h)

    crop_bbox = utils.expand_bbox_from_bounds(x_min, x_max, y_min, y_max, w, h, pad=0)

    cx1, cy1, cx2, cy2 = crop_bbox
    cx1, cy1 = max(0, cx1), max(0, cy1)
    cx2, cy2 = min(w, max(cx1 + 1, cx2)), min(h, max(cy1 + 1, cy2))
    return [cx1, cy1, cx2, cy2]


def remap_labels_to_crop(labels_cxcywhn, crop_bbox, src_w, src_h):
    """Map full-image labels to normalized coordinates in crop space."""
    cx1, cy1, cx2, cy2 = crop_bbox
    cw, ch = cx2 - cx1 + 1, cy2 - cy1 + 1
    out = []
    for cls_id, xc, yc, bw, bh in labels_cxcywhn:
        x1, y1, x2, y2 = utils.Bbox(
            cxcywhn=(xc, yc, bw, bh), frame_wh=(src_w, src_h)
        ).xyxy
        x1, y1, x2, y2 = max(x1, cx1), max(y1, cy1), min(x2, cx2), min(y2, cy2)
        if x2 <= x1 or y2 <= y1:
            continue
        x1, y1, x2, y2 = x1 - cx1, y1 - cy1, x2 - cx1, y2 - cy1
        nxc, nyc, nbw, nbh = utils.Bbox(
            xyxy=(x1, y1, x2, y2), frame_wh=(cw, ch)
        ).cxcywhn
        out.append((cls_id, nxc, nyc, nbw, nbh))
    return out


def write_labels(path, labels):
    """Write YOLO labels for one image to disk."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for cls_id, xc, yc, bw, bh in labels:
            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

In [ ]:
# Load models
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
suggestor_model = YOLO(str(PROJECT_ROOT / "models_staging" / "yolo26n_finetune.pt"))
embedding_preprocess = ResNet18_Weights.DEFAULT.transforms()
embedding_model = (
    torch.nn.Sequential(
        *list(resnet18(weights=ResNet18_Weights.DEFAULT).children())[:-1]
    )
    .to(device)
    .eval()
)

In [ ]:
# Load already-exported images and calculate their hashes and embeddings
existing_image_paths = sorted((EXPORT_ROOT / "images" / SOURCE_SPLITS[0]).glob("*.jpg"))
existing_image_paths += sorted(
    (EXPORT_ROOT / "images" / SOURCE_SPLITS[1]).glob("*.jpg")
)

existing_image_hashes = set()
existing_embeddings = []
existing_class_counts = {cls_id: 0 for cls_id in CLASS_NAMES}
for img_path in tqdm(existing_image_paths, desc="Indexing existing exports"):
    img = cv2.imread(str(img_path))
    existing_image_hashes.add(get_image_hash(img))
    existing_embeddings.append(get_embedding_cached(img))
    lbl_path = EXPORT_ROOT / "labels" / img_path.parent.name / f"{img_path.stem}.txt"
    for lbl in parse_yolo_txt(lbl_path):
        cls_id = int(lbl[0])
        existing_class_counts[cls_id] += 1

print(f"Existing class counts: {existing_class_counts}")

In [ ]:
# Load videos and extract candidate frames
video_paths = sorted(
    list(MOCK_INPUTS.glob("*_raw.avi")) + list(RAW_VIDEO.glob("*_raw.avi"))
)
raw_frames = []
frame_id = 1
for vpath in tqdm(video_paths, desc="Extracting frames"):
    cap = cv2.VideoCapture(str(vpath))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % 2 == 0:
            frame_hash = hashlib.sha1(frame.tobytes()).hexdigest()
            if frame_hash not in existing_image_hashes:
                raw_frames.append((frame_id, vpath.name, i, frame.copy()))
                frame_id += 1
        i += 1
    cap.release()

In [ ]:
# Calculate YOLO suggestions and embeddings for each candidate frame
candidates = []
for fid, vname, fidx, frame in tqdm(
    raw_frames, desc="YOLO Suggestion & Diversity Embeddings"
):

    # YOLO suggestion (cached)
    cached_pred = get_yolo_prediction_cached(
        frame,
        imgsz=tuple(settings.DETECTION_IMGSZ),
        conf=YOLO_CONF_FLOOR,
        max_det=settings.MAX_DETS,
    )
    labels = cached_pred["labels"]
    max_conf = cached_pred["max_conf"]
    uncertainty = 1.0 if len(labels) == 0 else (1.0 - max_conf)

    # diversity embeddings (cached)
    emb = get_embedding_cached(frame)

    # store candidate frame data
    candidates.append(
        Candidate(
            frame_id=fid,
            video_name=vname,
            frame_idx=fidx,
            image_bgr=frame,
            labels_cxcywhn=labels,
            uncertainty=uncertainty,
            embedding=emb,
        )
    )

In [ ]:
# Select diverse and difficult frames for labeling
emb_matrix = np.stack([c.embedding for c in candidates], axis=0)
uncert = np.array([c.uncertainty for c in candidates], dtype=np.float32)
existing_emb_matrix = (
    np.stack(existing_embeddings, axis=0) if len(existing_embeddings) > 0 else None
)

selected = []
remaining = set(range(len(candidates)))
class_counts = dict(existing_class_counts)
while len(selected) < TARGET_LABEL_FRAMES and remaining:
    best_idx = None
    best_score = -1.0
    for i in remaining:

        # calculate diversity to already-selected and existing embeddings
        reference_max_sim = 0.0
        if selected:
            selected_emb = emb_matrix[selected]
            reference_max_sim = max(
                reference_max_sim, np.max(selected_emb @ emb_matrix[i])
            )
        if existing_emb_matrix is not None:
            reference_max_sim = max(
                reference_max_sim, np.max(existing_emb_matrix @ emb_matrix[i])
            )
        diversity = np.clip(1.0 - reference_max_sim, 0.0, 1.0)

        # score based on uncertainty, diversity and class imbalance
        candidate_classes = {int(lbl[0]) for lbl in candidates[i].labels_cxcywhn}
        balance = (
            1
            - max(
                [
                    class_counts[c] / sum(class_counts.values())
                    for c in candidate_classes
                ]
            )
            if candidate_classes
            else 0.5
        )
        score = (
            ALPHA_UNCERTAINTY * uncert[i]
            + ALPHA_BALANCE * balance
            + (1 - ALPHA_UNCERTAINTY - ALPHA_BALANCE) * diversity
        )
        if score > best_score:
            best_score = score
            best_idx = i

    selected.append(best_idx)
    remaining.remove(best_idx)
    for lbl in candidates[best_idx].labels_cxcywhn:
        class_counts[int(lbl[0])] += 1

selected_candidates = [candidates[i] for i in selected]
print(f"Total class counts (existing + new):  {class_counts}")

In [ ]:
# Review selected frames and export after each interaction
for split in SOURCE_SPLITS:
    (EXPORT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (EXPORT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

next_id = (
    max(
        (
            int(p.stem)
            for split in SOURCE_SPLITS
            for p in (EXPORT_ROOT / "images" / split).glob("*.jpg")
            if p.stem.isdigit()
        ),
        default=0,
    )
    + 1
)

indices = list(range(len(selected_candidates)))
random.shuffle(indices)
n_train = int(round(TRAIN_SPLIT * len(indices)))
train_set = set(indices[:n_train])


def build_preview(img, labels, i, stem, split):
    preview = img.copy()

    for cls_id, xc, yc, bw, bh in labels:
        x1, y1, x2, y2 = utils.Bbox(
            cxcywhn=(xc, yc, bw, bh), frame_wh=tuple(reversed(img.shape[:2]))
        ).xyxy
        colour = utils.OBJECT_COLOUR_MAP[CLASS_NAMES[cls_id]]
        cv2.rectangle(preview, (x1, y1), (x2, y2), colour, 2)
        cv2.putText(
            preview,
            CLASS_NAMES.get(cls_id, str(cls_id)),
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            colour,
            1,
            cv2.LINE_AA,
        )

    info = f"{i + 1}/{len(selected_candidates)} id={stem} split={split} | r=redraw e=empty q=quit"
    cv2.putText(
        preview,
        info,
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 180, 255),
        2,
        cv2.LINE_AA,
    )
    return preview


for i, cand in enumerate(selected_candidates):
    split = SOURCE_SPLITS[0] if i in train_set else SOURCE_SPLITS[1]
    stem = f"{next_id + i:06d}"
    img_path = EXPORT_ROOT / "images" / split / f"{stem}.jpg"
    lbl_path = EXPORT_ROOT / "labels" / split / f"{stem}.txt"

    img = cand.image_bgr.copy()
    labels = []

    cv2.imshow("review", build_preview(img, labels, i, stem, split))
    key = cv2.waitKey(0) & 0xFF

    if key == ord("q"):
        break
    elif key == ord("e"):
        final_labels = []
    elif key == ord("r"):
        rois = draw_boxes_interactive(img, window_name="draw boxes")
        if rois is None or len(rois) == 0:
            continue
        final_labels = []
        for x1, y1, x2, y2, cls_id in rois:
            bbox = utils.Bbox(
                xyxy=(x1, y1, x2, y2), frame_wh=tuple(reversed(img.shape[:2]))
            )
            final_labels.append((cls_id, *bbox.cxcywhn))
    else:
        continue

    cv2.imwrite(str(img_path), img)

    if final_labels:
        with open(lbl_path, "w", encoding="utf-8") as f:
            for cls_id, xc, yc, bw, bh in final_labels:
                f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
# Crop labelled full frame fine tune data as in app
for split in SOURCE_SPLITS:

    # set up directories
    src_img_dir = EXPORT_ROOT / "images" / split
    src_lbl_dir = EXPORT_ROOT / "labels" / split
    dst_img_dir = CROPPED_EXPORT_ROOT / "images" / split
    dst_lbl_dir = CROPPED_EXPORT_ROOT / "labels" / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path in tqdm(sorted(src_img_dir.glob("*.jpg")), desc=f"Cropping {split}"):

        # identify paths and load image and labels
        stem = img_path.stem
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        labels = parse_yolo_txt(src_lbl_dir / f"{stem}.txt")

        if labels:

            # crop image and alter label
            bboxes = [
                utils.Bbox(cxcywhn=tuple(label[1:]), frame_wh=(w, h))
                for label in labels
            ]
            x_min = min(bbox.xyxy[0] for bbox in bboxes)
            y_min = min(bbox.xyxy[1] for bbox in bboxes)
            x_max = max(bbox.xyxy[2] for bbox in bboxes)
            y_max = max(bbox.xyxy[3] for bbox in bboxes)
            crop_bbox = utils.expand_bbox_from_bounds(
                x_min, x_max, y_min, y_max, w, h, 0.1
            ) or [0, 0, w, h]
            cropped_labels = remap_labels_to_crop(labels, crop_bbox, w, h)

        else:

            # randomly crop if no labels
            crop_bbox = random_aspect_crop_bbox(w, h)
            cropped_labels = []

        # save image and labels
        cx1, cy1, cx2, cy2 = crop_bbox
        cv2.imwrite(
            str(dst_img_dir / f"{stem}.jpg"), img[cy1 : cy2 + 1, cx1 : cx2 + 1].copy()
        )
        write_labels(dst_lbl_dir / f"{stem}.txt", cropped_labels)

In [ ]:
# Display original vs cropped new samples with labels

# load and subsample images
records = [
    (split, p.stem, p)
    for split in SOURCE_SPLITS
    for p in sorted((EXPORT_ROOT / "images" / split).glob("*.jpg"))
    if int(p.stem) >= next_id
]
sample = random.sample(records, min(8, len(records)))

# initialise figure and loop over sample/axes
fig, ax = plt.subplots(len(sample), 2, figsize=(15, 4 * len(sample)))
for (split, stem, src_img_path), ax_row in zip(sample, ax):

    # load images and labels
    src_lbl = parse_yolo_txt(EXPORT_ROOT / "labels" / split / f"{stem}.txt")
    crp_lbl = parse_yolo_txt(CROPPED_EXPORT_ROOT / "labels" / split / f"{stem}.txt")
    src = cv2.imread(str(src_img_path))
    crp = cv2.imread(str(CROPPED_EXPORT_ROOT / "images" / split / f"{stem}.jpg"))

    # plot annotated image
    ax_row[0].imshow(cv2.cvtColor(draw_cxcywhn(src, src_lbl), cv2.COLOR_BGR2RGB))
    ax_row[0].axis("off")
    ax_row[1].imshow(cv2.cvtColor(draw_cxcywhn(crp, crp_lbl), cv2.COLOR_BGR2RGB))
    ax_row[1].axis("off")

fig.tight_layout()

In [ ]:
# copy to final finetune dataset structure
DATASETS_ROOT = PROJECT_ROOT.parent / "datasets"
DST_FINETUNE = DATASETS_ROOT / "finetune"
DST_FINETUNE_WIDE = DATASETS_ROOT / "finetune_wide"
DST_FINETUNE_CROP = DATASETS_ROOT / "finetune_crop"


def copy_dataset_split(
    src_root: Path, dst_root: Path, split: str, stem_suffix: str = ""
):
    src_img_dir = src_root / "images" / split
    src_lbl_dir = src_root / "labels" / split
    dst_img_dir = dst_root / "images" / split
    dst_lbl_dir = dst_root / "labels" / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path in sorted(src_img_dir.glob("*.jpg")):
        stem = img_path.stem
        out_stem = f"{stem}{stem_suffix}"

        dst_img_path = dst_img_dir / f"{out_stem}.jpg"
        copy2(img_path, dst_img_path)

        src_lbl_path = src_lbl_dir / f"{stem}.txt"
        dst_lbl_path = dst_lbl_dir / f"{out_stem}.txt"
        if src_lbl_path.exists():
            copy2(src_lbl_path, dst_lbl_path)


for split in SOURCE_SPLITS:
    # non-cropped
    copy_dataset_split(EXPORT_ROOT, DST_FINETUNE_WIDE, split, stem_suffix="")
    copy_dataset_split(EXPORT_ROOT, DST_FINETUNE, split, stem_suffix="_wide")

    # cropped
    copy_dataset_split(CROPPED_EXPORT_ROOT, DST_FINETUNE_CROP, split, stem_suffix="")
    copy_dataset_split(CROPPED_EXPORT_ROOT, DST_FINETUNE, split, stem_suffix="_crop")